# 🚀 XAU/USD (Gold) FVG Detector - YOLOv8 Auto Trainer
Ye notebook XAU/USD 15-minute candlestick chart par **Bullish FVG** aur **Bearish FVG** detect karne ke liye YOLOv8 model train karega.

### ⚡ Shuru Karne Se Pehle:
1. Upar menu mein jao: **Runtime > Change runtime type**
2. Hardware accelerator: **T4 GPU** select karo aur **Save** dabao.
3. Upar menu se **Runtime > Run all** (`Ctrl + F9`) daba do.

### Step 1: Install Libraries & Load XAUUSD Historical Data

In [ ]:
!pip install -q mplfinance ultralytics huggingface_hub gradio

import os, shutil
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import mplfinance as mpf
from huggingface_hub import list_repo_files, hf_hub_download

plt.ioff()
REPO = "SpatialGhost/xauusd-gold-price-historical-data-2004-2025"
START_YEAR = "2023"

print("⏳ Hugging Face se dataset dhund rahe hain...")
files = [f for f in list_repo_files(REPO, repo_type="dataset")
         if f.lower().endswith((".csv", ".json", ".jsonl", ".parquet"))]
print("Available Files:", files)
pick = [f for f in files if "15" in os.path.basename(f)]
FILE = pick[0] if pick else files[0]
print("Downloading file:", FILE)
path = hf_hub_download(REPO, FILE, repo_type="dataset")

def prep(c):
    c = c.rename(columns=lambda x: str(x).strip().capitalize())
    c = c.rename(columns={"Time": "Date", "Datetime": "Date", "Timestamp": "Date"})
    return c[c["Date"].astype(str) >= START_YEAR]

chunks = []
if path.endswith(".parquet"):
    chunks = [prep(pd.read_parquet(path))]
elif path.endswith(".csv"):
    head = open(path, encoding="utf-8", errors="ignore").readline()
    sep = ";" if head.count(";") > head.count(",") else ","
    for c in pd.read_csv(path, sep=sep, chunksize=500000):
        chunks.append(prep(c))
else:
    try:
        for c in pd.read_json(path, lines=True, chunksize=500000):
            chunks.append(prep(c))
    except ValueError:
        chunks = [prep(pd.read_json(path))]

df = pd.concat(chunks)
d = pd.to_datetime(df["Date"], format="%Y.%m.%d %H:%M", errors="coerce")
if d.isna().mean() > 0.5:
    d = pd.to_datetime(df["Date"], errors="coerce")
df["Date"] = d
df = df.dropna(subset=["Date"])
if "Volume" not in df.columns:
    df["Volume"] = 0
for col in ["Open", "High", "Low", "Close", "Volume"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
df = df.set_index("Date").sort_index()
df = df[~df.index.duplicated()][["Open", "High", "Low", "Close", "Volume"]].dropna()

gap = df.index.to_series().diff().median()
if gap < pd.Timedelta("15min"):
    df = df.resample("15min").agg({"Open": "first", "High": "max", "Low": "min",
                                   "Close": "last", "Volume": "sum"}).dropna()
print(f"✅ Data ready! Candle interval: {gap} | Total Rows: {len(df)}")
assert len(df) > 1000, "Data kam hai, please check dataset."
df.tail()

### Step 2: Auto-labeling FVG Zones & Generating Dataset Images

In [ ]:
shutil.rmtree("ds", ignore_errors=True)
for s in ["train", "val"]:
    os.makedirs(f"ds/images/{s}", exist_ok=True)
    os.makedirs(f"ds/labels/{s}", exist_ok=True)

W, STEP, DPI, MAX_IMAGES = 100, 50, 100, 1500

def find_zones(w):
    hi, lo = w.High.values, w.Low.values
    min_size = (w.High - w.Low).mean() * 0.5
    z = []
    for i in range(2, len(w)):
        if lo[i] - hi[i-2] > min_size: z.append((0, i-2, hi[i-2], lo[i]))   # 0: bullish FVG
        if lo[i-2] - hi[i] > min_size: z.append((1, i-2, hi[i], lo[i-2]))   # 1: bearish FVG
    return z

starts = list(range(0, len(df) - W, STEP))[-MAX_IMAGES:]
split_at = int(len(starts) * 0.8)
saved = 0
print("⏳ Dataset images generate ho rahi hain...")
for k, st in enumerate(starts):
    w = df.iloc[st:st+W]
    zones = find_zones(w)
    if not zones:
        continue
    s = "train" if k < split_at else "val"
    fig, axes = mpf.plot(w, type="candle", style="charles", axisoff=True,
                         figsize=(8, 8), returnfig=True)
    fig.set_dpi(DPI)
    ax = axes[0]
    fig.canvas.draw()
    H = 8 * DPI
    lines = []
    for c, i0, zl, zh in zones:
        x0, y0 = ax.transData.transform((i0 - 0.5, zl))
        x1, y1 = ax.transData.transform((min(i0 + 8, W - 1) + 0.5, zh))
        lines.append(f"{c} {(x0+x1)/2/H:.6f} {1-(y0+y1)/2/H:.6f} {abs(x1-x0)/H:.6f} {abs(y1-y0)/H:.6f}")
    fig.savefig(f"ds/images/{s}/{k}.png", dpi=DPI)
    plt.close(fig)
    with open(f"ds/labels/{s}/{k}.txt", "w") as f:
        f.write("\n".join(lines))
    saved += 1
    if saved % 200 == 0:
        print(f"Generated images: {saved}")

with open("ds/data.yaml", "w") as f:
    f.write("path: /content/ds\ntrain: images/train\nval: images/val\nnames:\n  0: bull_fvg\n  1: bear_fvg\n")

print(f"✅ Dataset Tayyar Hai! Total Images Saved: {saved}")
assert saved > 50, "Images kam bane, please data check karein."

### Step 3: Train YOLOv8 Model (GPU Par ~10-15 Min)

In [ ]:
from ultralytics import YOLO

print("🚀 YOLOv8 Training shuru ho rahi hai...")
model = YOLO("yolov8n.pt")
results = model.train(
    data="/content/ds/data.yaml",
    epochs=30,
    imgsz=640,
    batch=16,
    project="/content/runs",
    name="fvg",
    exist_ok=True,
    plots=True
)

best = model.trainer.best
print("🎉 Training Complete! BEST MODEL PATH:", best)

### Step 4: Test Model on Validation Charts

In [ ]:
import glob, cv2
from google.colab.patches import cv2_imshow

imgs = sorted(glob.glob("/content/ds/images/val/*.png"))[:3]
print("Validation samples par test results:")
for p in imgs:
    r = YOLO(best)(p, conf=0.4, verbose=False)
    cv2_imshow(r[0].plot())

### Step 5: Save Model to Google Drive & Download

In [ ]:
from google.colab import files

# Google Drive mein backup (optional but recommended)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs('/content/drive/MyDrive/chart_model', exist_ok=True)
    shutil.copy(best, '/content/drive/MyDrive/chart_model/best.pt')
    print("✅ Google Drive mein 'chart_model/best.pt' save ho gaya!")
except Exception as e:
    print("Drive mount skip hua ya error:", e)

# Seedhe PC par download prompt
print("📥 Model download ho raha hai aapke PC par...")
files.download(best)

### Step 6: Interactive Web App (Gradio) - Live Detection & Analysis

In [ ]:
import gradio as gr
from ultralytics import YOLO

fvg_model = YOLO(best)

def make_report(zones):
    if not zones:
        return "⚠️ Koi clear FVG zone detect nahi hua. Kripya 15-minute candle chart bina indicators/grid ke upload karein."
    zones = sorted(zones, key=lambda z: z["right_pct"], reverse=True)
    bull = [z for z in zones if z["type"] == "bull_fvg"]
    bear = [z for z in zones if z["type"] == "bear_fvg"]
    last = zones[0]
    side = "Bullish" if last["type"] == "bull_fvg" else "Bearish"
    pos = "upar ke hisse" if last["top_pct"] < 50 else "neeche ke hisse"
    recent = zones[:3]
    rb = sum(z["type"] == "bull_fvg" for z in recent)
    bias = "Bullish" if rb >= 2 else "Bearish" if rb <= 1 else "Mixed / Neutral"
    return (
        f"📊 **Detection Summary**:\n"
        f"- Total Detected Zones: {len(zones)}\n"
        f"- 🟢 Bullish FVGs: {len(bull)}\n"
        f"- 🔴 Bearish FVGs: {len(bear)}\n\n"
        f"🎯 **Latest Formation**: {side} FVG chart ke {pos} mein bana hai (Confidence: {last['conf'] * 100:.1f}%).\n"
        f"🧭 **Short-term Bias**: {bias}\n\n"
        f"ℹ️ **Note**: FVG imbalances liquidity fill hone par reaction de sakte hain. Confirm structure before entry."
    )

def analyze_chart(img):
    if img is None:
        return None, "Kripya image upload karein."
    img_bgr = img[:, :, :3][:, :, ::-1]
    r = fvg_model(img_bgr, conf=0.35, verbose=False)[0]
    h, w = r.orig_shape
    zones = []
    for b in r.boxes:
        x1, y1, x2, y2 = b.xyxy[0].tolist()
        zones.append({
            "type": r.names[int(b.cls)],
            "conf": round(float(b.conf), 2),
            "right_pct": round(x2 / w * 100),
            "top_pct": round(y1 / h * 100)
        })
    res_img = r.plot()[:, :, ::-1]
    return res_img, make_report(zones)

demo = gr.Interface(
    fn=analyze_chart,
    inputs=gr.Image(type="numpy", label="Chart Upload Karo (XAU/USD 15m)"),
    outputs=[
        gr.Image(label="Detected FVGs"),
        gr.Markdown(label="Analysis Report")
    ],
    title="📈 Gold (XAU/USD) FVG AI Detector",
    description="Upload any 15m candlestick chart to detect Bullish and Bearish Fair Value Gaps (FVG) automatically."
)

demo.launch(share=True, debug=True)
